# 🚀 ASTRA — Adaptive Semantic Task Reasoning Architecture
### DVCon India 2026 Design Contest — Stage 2A Implementation
**Team:** KAMALESH S, KESHAV J, RAGHUL V | VILSCORE, 389 | Saveetha Engineering College

---
**Pipeline:** YOLOv5s → CLIP Encoding → Knowledge Graph → Scene Context → Weighted Scorer → Best Object

**14 Official Tasks (COCO-Tasks paper, arXiv:1904.03000):**
1. step on something to reach top of a shelf
2. sit comfortably
3. place flowers
4. get potatoes out of fire
5. water a plant
6. get lemon out of tea
7. dig a hole
8. open a bottle of beer
9. open a parcel
10. serve wine
11. pour sugar
12. smear butter
13. extinguish fire
14. pound a carpet

> **Stage 2A requirement:** CPU inference only for all pipeline modules.


## CELL 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1 — INSTALL ALL DEPENDENCIES
# ============================================================
# Run this cell FIRST. Expected time: ~3-4 minutes on Colab.

!pip install ultralytics --quiet
!pip install git+https://github.com/openai/CLIP.git --quiet
!pip install networkx --quiet
!pip install pycocotools --quiet
!pip install opencv-python-headless --quiet
!pip install matplotlib pillow numpy torch torchvision --quiet

print("✅ All dependencies installed!")


## CELL 2 — Imports & Verify

In [ ]:
# ============================================================
# CELL 2 — IMPORTS AND VERIFICATION
# ============================================================
import os, json, math, time, urllib.request, warnings, zipfile
warnings.filterwarnings('ignore')

import numpy as np
import torch
import clip
import cv2
import networkx as nx
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from IPython.display import display, clear_output

# ── Stage 2A: CPU inference enforced ──
INFERENCE_DEVICE = "cpu"

print("✅ All imports successful!")
print(f"   PyTorch       : {torch.__version__}")
print(f"   CLIP          : {'available'}")
print(f"   Inference on  : {INFERENCE_DEVICE.upper()}  (Stage 2A requirement)")


## CELL 3 — ASTRA Knowledge Graph (14 Official Tasks)

In [ ]:
# ============================================================
# CELL 3 — TASK-OBJECT KNOWLEDGE GRAPH
# ============================================================
# Manually constructed from COCO-Tasks paper Table 1.
# Tasks are EXACTLY as in: Sawatzky et al., arXiv:1904.03000
#
# Weight semantics:
#   1.0 = canonical / ideal object for task
#   0.8 = strong alternative
#   0.6 = acceptable fallback
#   0.4 = last resort
#
# All object labels are COCO-80 lowercase strings (as returned by YOLOv5).

class TaskObjectGraph:
    """
    ASTRA Knowledge Graph Module.
    Directed weighted graph: task → COCO object.
    Implements Sg (graph relevance score) in: Sf = α·Ci + β·Sg + γ·Ss
    """

    TASK_KEYS = [
        "step on something to reach top of a shelf",  # T1
        "sit comfortably",                             # T2
        "place flowers",                               # T3
        "get potatoes out of fire",                    # T4
        "water a plant",                               # T5
        "get lemon out of tea",                        # T6
        "dig a hole",                                  # T7
        "open a bottle of beer",                       # T8
        "open a parcel",                               # T9
        "serve wine",                                  # T10
        "pour sugar",                                  # T11
        "smear butter",                                # T12
        "extinguish fire",                             # T13
        "pound a carpet",                              # T14
    ]

    def __init__(self):
        self.graph = nx.DiGraph()
        self._build_graph()

    def _build_graph(self):
        edges = [
            # T1 — step on something to reach top of a shelf
            ("step on something to reach top of a shelf", "chair",        1.0),
            ("step on something to reach top of a shelf", "bench",        0.9),
            ("step on something to reach top of a shelf", "dining table", 0.8),
            ("step on something to reach top of a shelf", "couch",        0.6),
            ("step on something to reach top of a shelf", "bed",          0.5),
            ("step on something to reach top of a shelf", "suitcase",     0.5),
            ("step on something to reach top of a shelf", "refrigerator", 0.4),
            ("step on something to reach top of a shelf", "toilet",       0.3),
            ("step on something to reach top of a shelf", "skateboard",   0.3),
            ("step on something to reach top of a shelf", "book",         0.2),
            # T2 — sit comfortably
            ("sit comfortably", "couch",  1.0),
            ("sit comfortably", "chair",  0.9),
            ("sit comfortably", "bench",  0.8),
            ("sit comfortably", "bed",    0.7),
            ("sit comfortably", "toilet", 0.3),
            # T3 — place flowers
            ("place flowers", "vase",       1.0),
            ("place flowers", "cup",        0.7),
            ("place flowers", "wine glass", 0.6),
            ("place flowers", "bottle",     0.5),
            ("place flowers", "bowl",       0.4),
            # T4 — get potatoes out of fire
            ("get potatoes out of fire", "fork",          1.0),
            ("get potatoes out of fire", "knife",         0.8),
            ("get potatoes out of fire", "spoon",         0.7),
            ("get potatoes out of fire", "bowl",          0.5),
            ("get potatoes out of fire", "scissors",      0.4),
            ("get potatoes out of fire", "baseball bat",  0.3),
            ("get potatoes out of fire", "tennis racket", 0.2),
            ("get potatoes out of fire", "umbrella",      0.2),
            # T5 — water a plant
            ("water a plant", "bottle",     1.0),
            ("water a plant", "cup",        0.8),
            ("water a plant", "bowl",       0.6),
            ("water a plant", "wine glass", 0.5),
            ("water a plant", "vase",       0.4),
            # T6 — get lemon out of tea
            ("get lemon out of tea", "spoon",    1.0),
            ("get lemon out of tea", "fork",     0.8),
            ("get lemon out of tea", "knife",    0.5),
            ("get lemon out of tea", "scissors", 0.3),
            # T7 — dig a hole
            ("dig a hole", "baseball bat",  0.9),
            ("dig a hole", "knife",         0.7),
            ("dig a hole", "fork",          0.5),
            ("dig a hole", "umbrella",      0.4),
            ("dig a hole", "scissors",      0.4),
            ("dig a hole", "tennis racket", 0.3),
            ("dig a hole", "skis",          0.3),
            ("dig a hole", "surfboard",     0.2),
            ("dig a hole", "snowboard",     0.2),
            # T8 — open a bottle of beer
            ("open a bottle of beer", "knife",     1.0),
            ("open a bottle of beer", "scissors",  0.7),
            ("open a bottle of beer", "fork",      0.4),
            ("open a bottle of beer", "spoon",     0.3),
            # T9 — open a parcel
            ("open a parcel", "scissors", 1.0),
            ("open a parcel", "knife",    0.9),
            ("open a parcel", "fork",     0.3),
            # T10 — serve wine
            ("serve wine", "wine glass", 1.0),
            ("serve wine", "cup",        0.7),
            ("serve wine", "bottle",     0.4),
            ("serve wine", "bowl",       0.3),
            # T11 — pour sugar
            ("pour sugar", "spoon",  1.0),
            ("pour sugar", "bowl",   0.9),
            ("pour sugar", "cup",    0.7),
            ("pour sugar", "bottle", 0.4),
            ("pour sugar", "fork",   0.3),
            # T12 — smear butter
            ("smear butter", "knife",  1.0),
            ("smear butter", "spoon",  0.7),
            ("smear butter", "fork",   0.5),
            # T13 — extinguish fire
            ("extinguish fire", "fire hydrant", 1.0),
            ("extinguish fire", "bottle",       0.6),
            ("extinguish fire", "bowl",         0.4),
            ("extinguish fire", "cup",          0.3),
            # T14 — pound a carpet
            ("pound a carpet", "baseball bat",  1.0),
            ("pound a carpet", "tennis racket", 0.8),
            ("pound a carpet", "umbrella",      0.6),
            ("pound a carpet", "skis",          0.5),
            ("pound a carpet", "frisbee",       0.3),
            ("pound a carpet", "surfboard",     0.3),
            ("pound a carpet", "snowboard",     0.2),
        ]
        for task, obj, w in edges:
            self.graph.add_edge(task, obj, weight=w)

    def get_graph_score(self, task: str, object_label: str) -> float:
        t = task.lower().strip()
        o = object_label.lower().strip()
        if self.graph.has_edge(t, o):
            return self.graph[t][o]['weight']
        # Partial substring match with discount
        for node in self.graph.successors(t):
            if node in o or o in node:
                return self.graph[t][node]['weight'] * 0.8
        return 0.0

    def get_preferred_objects(self, task: str):
        t = task.lower().strip()
        if t not in self.graph:
            return []
        return sorted(
            [(n, self.graph[t][n]['weight']) for n in self.graph.successors(t)],
            key=lambda x: x[1], reverse=True
        )

    def get_scene_boost(self, task: str, all_labels: list) -> dict:
        """
        Scene Context Reasoning:
        If a higher-ranked object for this task is absent from the scene,
        boost the scores of lower-ranked alternatives.
        Returns: {label: boost_multiplier}
        """
        preferred = self.get_preferred_objects(task)
        boosts = {}
        all_lower = [l.lower() for l in all_labels]
        # Find the rank of the best object actually present in the scene
        best_present_rank = None
        for rank, (obj, _) in enumerate(preferred):
            if obj in all_lower:
                best_present_rank = rank
                break
        # If top-ranked objects are missing, boost present fallbacks
        if best_present_rank is not None and best_present_rank > 0:
            for obj, w in preferred:
                if obj in all_lower:
                    # Boost proportional to how much the ideal is missing
                    boosts[obj] = 1.0 + (best_present_rank * 0.15)
        return boosts


# ── Instantiate and verify ──
kg = TaskObjectGraph()
print("✅ Knowledge Graph built!")
print(f"   Nodes : {kg.graph.number_of_nodes()}")
print(f"   Edges : {kg.graph.number_of_edges()}")
print()
print("Sample lookups:")
for t, o, expected in [
    ("serve wine", "wine glass", "HIGH"),
    ("serve wine", "cup",        "MED"),
    ("extinguish fire", "fire hydrant", "HIGH"),
    ("sit comfortably", "couch", "HIGH"),
    ("smear butter", "knife",    "HIGH"),
]:
    sg = kg.get_graph_score(t, o)
    print(f"  {t!r:45s} → {o!r:20s}  Sg={sg:.2f}  [{expected}]")


## CELL 4 — Object Detector (YOLOv5s, CPU)

In [ ]:
# ============================================================
# CELL 4 — OBJECT DETECTOR (YOLOv5s, CPU inference)
# ============================================================
# YOLOv5s pre-trained on COCO-80. Weights ~14 MB, auto-downloaded.
# conf_threshold=0.20 used for better recall on difficult scenes.

class ObjectDetector:
    """
    ASTRA Object Detection Module.
    Wraps YOLOv5s (Ultralytics) for structured CPU inference.
    Outputs per object: label, Ci (confidence), bbox [x1,y1,x2,y2], crop (RGB).
    """

    def __init__(self, model_name='yolov5s', conf_threshold=0.20, device='cpu'):
        print(f"  Loading {model_name} on {device.upper()}...")
        self.model          = YOLO(model_name + '.pt')
        self.conf_threshold = conf_threshold
        self.device         = device
        print("  ✅ Detector ready.")

    def detect(self, image_input, conf_threshold=None):
        """
        Args:
            image_input : str (path) or np.ndarray (RGB)
            conf_threshold : override default threshold

        Returns:
            detected_objects : list[dict]  — sorted by confidence desc
            image_rgb        : np.ndarray
        """
        conf = conf_threshold if conf_threshold is not None else self.conf_threshold

        if isinstance(image_input, str):
            bgr = cv2.imread(image_input)
            if bgr is None:
                raise FileNotFoundError(f"Image not found: {image_input}")
            image_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            image_rgb = np.array(image_input)

        results = self.model(image_rgb, conf=conf, device=self.device, verbose=False)

        h, w = image_rgb.shape[:2]
        detected = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2),  min(h, y2)
                crop = image_rgb[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                detected.append({
                    'label'     : r.names[int(box.cls[0])],
                    'confidence': float(box.conf[0]),
                    'bbox'      : [x1, y1, x2, y2],
                    'crop'      : crop,
                    'class_id'  : int(box.cls[0]),
                })

        detected.sort(key=lambda x: x['confidence'], reverse=True)
        return detected, image_rgb

    def detect_with_fallback(self, image_input, low_threshold=0.10):
        """
        If standard detection finds 0 objects, retry with lower threshold.
        Ensures pipeline never fails silently on difficult images.
        """
        objs, img = self.detect(image_input)
        if not objs:
            objs, img = self.detect(image_input, conf_threshold=low_threshold)
        return objs, img


detector = ObjectDetector(model_name='yolov5s', conf_threshold=0.20, device='cpu')
print("\n✅ ObjectDetector module ready!")


## CELL 5 — CLIP Encoder (Vision-Language Embeddings)

In [ ]:
# ============================================================
# CELL 5 — CLIP ENCODER
# ============================================================
# CLIP ViT-B/32: shared 512-dim embedding space for images and text.
# Implements:
#   Task Encoding   → Te  (Stage 3)
#   Object Encoding → Ei  (Stage 4)
#   Cosine Sim      → Ss  (Stage 6)
# All inference on CPU as required by Stage 2A.

class CLIPEncoder:
    """
    ASTRA Vision-Language Embedding Module.
    Dual-channel object encoding: text label + visual crop.
    Dual-channel task encoding:  direct prompt + action-framed prompt.
    """

    def __init__(self, model_name='ViT-B/32', device='cpu'):
        print(f"  Loading CLIP {model_name} on {device.upper()}...")
        self.device = device
        self.model, self.preprocess = clip.load(model_name, device=device)
        self.model.eval()
        print("  ✅ CLIP ready.")

    def encode_text(self, text: str) -> np.ndarray:
        """Encodes text → L2-normalised 512-dim vector (Te or Ei-text)."""
        with torch.no_grad():
            tok = clip.tokenize([text], truncate=True).to(self.device)
            f   = self.model.encode_text(tok)
            f   = f / f.norm(dim=-1, keepdim=True)
        return f.cpu().numpy()[0]

    def encode_image_crop(self, crop: np.ndarray) -> np.ndarray:
        """Encodes an RGB numpy crop → L2-normalised 512-dim vector (Ei-visual)."""
        if crop is None or crop.size == 0:
            return np.zeros(512, dtype=np.float32)
        try:
            pil = Image.fromarray(crop)
            with torch.no_grad():
                t = self.preprocess(pil).unsqueeze(0).to(self.device)
                f = self.model.encode_image(t)
                f = f / f.norm(dim=-1, keepdim=True)
            return f.cpu().numpy()[0]
        except Exception:
            return np.zeros(512, dtype=np.float32)

    def encode_label_as_text(self, label: str) -> np.ndarray:
        """CLIP best-practice prompt: 'a photo of a {label}'."""
        return self.encode_text(f"a photo of a {label}")

    def encode_task_direct(self, task: str) -> np.ndarray:
        """Direct task encoding."""
        return self.encode_text(task)

    def encode_task_as_action(self, task: str) -> np.ndarray:
        """Action-framed task encoding for better CLIP grounding."""
        return self.encode_text(f"an image showing someone who wants to {task}")

    def get_task_embedding(self, task: str) -> np.ndarray:
        """Ensemble task embedding: average of direct + action-framed."""
        te1 = self.encode_task_direct(task)
        te2 = self.encode_task_as_action(task)
        te  = (te1 + te2) / 2.0
        norm = np.linalg.norm(te)
        return te / (norm + 1e-8)

    def get_object_embedding(self, label: str, crop: np.ndarray) -> np.ndarray:
        """Dual-channel object embedding: average of text label + visual crop."""
        ei_text  = self.encode_label_as_text(label)
        ei_image = self.encode_image_crop(crop)
        if np.linalg.norm(ei_image) < 1e-8:
            return ei_text
        ei = (ei_text + ei_image) / 2.0
        norm = np.linalg.norm(ei)
        return ei / (norm + 1e-8)

    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Cosine similarity ∈ [-1, 1]."""
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na < 1e-8 or nb < 1e-8:
            return 0.0
        return float(np.dot(a, b) / (na * nb + 1e-8))


encoder = CLIPEncoder(model_name='ViT-B/32', device=INFERENCE_DEVICE)
print()

# ── Sanity test ──
te   = encoder.get_task_embedding("serve wine")
e_g  = encoder.encode_label_as_text("wine glass")
e_c  = encoder.encode_label_as_text("car")
print("✅ CLIPEncoder ready!")
print(f"   'serve wine' vs 'wine glass' : {CLIPEncoder.cosine_similarity(te, e_g):.4f}  (expect HIGH)")
print(f"   'serve wine' vs 'car'        : {CLIPEncoder.cosine_similarity(te, e_c):.4f}  (expect LOW)")


## CELL 6 — Scene Context Reasoning Module

In [ ]:
# ============================================================
# CELL 6 — SCENE CONTEXT REASONING MODULE
# ============================================================
# Implements the scene context stage from the ASTRA proposal.
#
# Two mechanisms:
#   A. Fallback boost  — if the best-ranked object for a task
#      is absent, boost scores of present alternatives.
#   B. Co-presence bonus — objects whose COCO category commonly
#      appears alongside task-relevant objects get a small boost
#      (e.g., "bottle" and "wine glass" co-occur → confirms dining scene).
#
# NEW (formula extension):
#   compute_scene_score() → Sp in [0, 1]
#   Used as additive term δ·Sp in: Sf = α·Ci + β·Sg + γ·Ss + δ·Sp

class SceneContextReasoner:
    """
    ASTRA Scene Context Reasoning Module.
    Refines per-object scores by analysing the global scene composition.
    Sp (scene proximity score) is the additive scene context component
    in the extended ASTRA scoring formula:
        Sf = α·Ci + β·Sg + γ·Ss + δ·Sp
    """

    # Co-presence patterns: task → set of reinforcing co-present categories
    CO_PRESENCE = {
        "serve wine"                              : {"bottle", "cup", "dining table", "bowl", "fork"},
        "sit comfortably"                         : {"tv", "laptop", "remote", "book"},
        "place flowers"                           : {"potted plant", "vase"},
        "get potatoes out of fire"                : {"bowl", "dining table", "oven", "microwave"},
        "water a plant"                           : {"potted plant", "vase", "sink"},
        "get lemon out of tea"                    : {"cup", "bowl", "dining table"},
        "dig a hole"                              : {"person", "backpack"},
        "open a bottle of beer"                   : {"bottle", "cup", "dining table"},
        "open a parcel"                           : {"book", "backpack", "suitcase"},
        "pour sugar"                              : {"cup", "dining table", "bowl", "spoon"},
        "smear butter"                            : {"knife", "dining table", "bowl"},
        "extinguish fire"                         : {"fire hydrant", "bottle"},
        "pound a carpet"                          : {"chair", "couch"},
        "step on something to reach top of a shelf": {"book", "backpack"},
    }

    def compute_scene_score(self, task: str, obj_label: str,
                             all_labels: list, graph) -> float:
        """
        Compute Sp (scene proximity score) for one object ∈ [0, 1].

        Two sub-components:
          1. Co-presence reinforcement (0.0 – 0.50):
             How many scene-context reinforcers are present in the image?
             E.g. "dining table" + "fork" confirm a dining scene for
             "serve wine", boosting "wine glass".
          2. Fallback positional boost (0.0 – 0.40):
             When the highest-ranked graph object is absent, lower-ranked
             alternatives that ARE present receive a context reward
             proportional to how many ideal objects are missing.

        Total Sp = co_presence_sp + fallback_sp, clipped to [0, 1].
        """
        sp         = 0.0
        task_l     = task.lower().strip()
        lbl        = obj_label.lower().strip()
        all_lower  = [l.lower() for l in all_labels]

        # ── Component 1: Co-presence reinforcement ──────────────────
        co_cats        = self.CO_PRESENCE.get(task_l, set())
        present_co     = co_cats & set(all_lower)
        # Up to 0.50: each co-present reinforcer adds 0.10 (max 5 count)
        co_sp          = min(0.50, len(present_co) * 0.10)
        # Only award co-presence bonus to objects with Sg > 0.0
        # (irrelevant objects should not benefit from scene confirmation)
        if graph.get_graph_score(task_l, lbl) > 0.0:
            sp += co_sp

        # ── Component 2: Fallback positional boost ──────────────────
        preferred = graph.get_preferred_objects(task_l)
        best_present_rank = None
        for rank, (obj, _) in enumerate(preferred):
            if obj in all_lower:
                best_present_rank = rank
                break

        # Award fallback boost only to objects that are:
        #   • actually present in the scene (in all_lower)
        #   • in the graph (Sg > 0)
        #   • and the ideal (#0) object is absent (best_present_rank > 0)
        if (best_present_rank is not None and best_present_rank > 0
                and lbl in all_lower
                and graph.get_graph_score(task_l, lbl) > 0.0):
            # Boost = 0.10 per missing top-ranked object (max 0.40)
            fallback_sp = min(0.40, best_present_rank * 0.10)
            sp += fallback_sp

        return round(min(1.0, sp), 4)

    def compute_context_scores(self, task: str, ranked_objects: list,
                                graph) -> list:
        """
        Attaches 'scene_score' (Sp) to each object dict.
        context_boost is kept as 1.0 (identity) since Sp is now
        an additive term in the main formula rather than a multiplier.
        """
        all_labels = [o['label'] for o in ranked_objects]
        updated = []
        for obj in ranked_objects:
            sp = self.compute_scene_score(task, obj['label'], all_labels, graph)
            updated.append({**obj, 'scene_score': sp, 'context_boost': 1.0})
        return updated

    def apply_boosts(self, ranked_objects: list) -> list:
        """Re-sort by final_score (context_boost is 1.0 — no change)."""
        ranked_objects.sort(key=lambda x: x['final_score'], reverse=True)
        return ranked_objects


scene_reasoner = SceneContextReasoner()
print("✅ SceneContextReasoner ready!")
print(f"   Tasks with co-presence rules : {len(SceneContextReasoner.CO_PRESENCE)}")
print()
print("Sp demo for 'serve wine' scene containing [wine glass, cup, bottle, fork]:")
demo_objs = ["wine glass", "cup", "bottle", "fork", "knife", "car"]
for obj in demo_objs:
    sp = scene_reasoner.compute_scene_score("serve wine", obj, demo_objs, kg)
    print(f"   {obj:<20}  Sp = {sp:.3f}")


## CELL 7 — Scoring Engine

In [ ]:
# ============================================================
# CELL 7 — ASTRA SCORING ENGINE
# ============================================================
# Extended ASTRA formula (Stage 2A addition):
#
#   Sf = α·Ci + β·Sg + γ·Ss + δ·Sp
#
#   Ci = YOLOv5 detection confidence           (α = 0.20)
#   Sg = Knowledge graph relevance score       (β = 0.40)  ← highest
#   Ss = CLIP cosine similarity (normalised)   (γ = 0.25)
#   Sp = Scene proximity / context score       (δ = 0.15)  ← NEW
#
# α + β + γ + δ = 0.20 + 0.40 + 0.25 + 0.15 = 1.00
#
# Sp is computed by SceneContextReasoner.compute_scene_score()
# and is an ADDITIVE term — no post-hoc multiplier applied.

class ScoringEngine:
    """
    ASTRA Final Scoring Module.
    Implements Sf = α·Ci + β·Sg + γ·Ss + δ·Sp
    where Sp is the scene proximity score from SceneContextReasoner.
    """

    def __init__(self, alpha=0.20, beta=0.40, gamma=0.25, delta=0.15):
        assert abs(alpha + beta + gamma + delta - 1.0) < 1e-6, \
            "Weights α+β+γ+δ must sum to 1.0"
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma
        self.delta = delta

    def compute_sf(self, ci: float, sg: float, ss: float, sp: float) -> float:
        """
        Sf = α·Ci + β·Sg + γ·Ss_norm + δ·Sp
        Ss is normalised from [-1, 1] → [0, 1] before weighting.
        """
        ss_norm = (ss + 1.0) / 2.0          # normalise [-1,1] → [0,1]
        return (self.alpha * ci
                + self.beta  * sg
                + self.gamma * ss_norm
                + self.delta * sp)

    def rank_objects(self, detected_objects, task, graph, enc, scene_ctx) -> list:
        """
        Full ranking pipeline:
          1. Encode task → Te  (ensemble)
          2. For each object: get Ci, Sg, Ss, Sp → compute Sf
          3. Apply scene context (Sp already baked in; boost = 1.0)
          4. Return sorted list (O* first)
        """
        te         = enc.get_task_embedding(task)
        all_labels = [o['label'] for o in detected_objects]

        scored = []
        for obj in detected_objects:
            label = obj['label']
            ci    = obj['confidence']
            sg    = graph.get_graph_score(task, label)
            ei    = enc.get_object_embedding(label, obj.get('crop'))
            ss    = CLIPEncoder.cosine_similarity(te, ei)
            sp    = scene_ctx.compute_scene_score(task, label, all_labels, graph)
            sf    = self.compute_sf(ci, sg, ss, sp)

            scored.append({
                **obj,
                'graph_score'    : round(sg, 4),
                'semantic_sim'   : round(ss, 4),
                'scene_score'    : round(sp, 4),
                'final_score'    : round(sf, 6),
                'context_boost'  : 1.0,
                'score_breakdown': {
                    'alpha_Ci' : round(self.alpha * ci,               4),
                    'beta_Sg'  : round(self.beta  * sg,               4),
                    'gamma_Ss' : round(self.gamma * ((ss + 1) / 2),   4),
                    'delta_Sp' : round(self.delta * sp,               4),
                    'Sf_raw'   : round(sf,                            4),
                }
            })

        scored.sort(key=lambda x: x['final_score'], reverse=True)

        # ── Attach context metadata (boost=1.0; Sp already in formula) ──
        scored = scene_ctx.compute_context_scores(task, scored, graph)
        scored = scene_ctx.apply_boosts(scored)
        return scored


scorer = ScoringEngine(alpha=0.20, beta=0.40, gamma=0.25, delta=0.15)
print("✅ ScoringEngine ready!")
print(f"   Sf = {scorer.alpha}·Ci + {scorer.beta}·Sg + {scorer.gamma}·Ss + {scorer.delta}·Sp")
print(f"   Weight sum = {scorer.alpha+scorer.beta+scorer.gamma+scorer.delta:.2f}")
print()
print("Formula demo (theoretical values for 'serve wine'):")
for label, ci, sg, ss, sp in [
    ("wine glass", 0.92, 1.00, 0.85, 0.50),
    ("cup",        0.78, 0.70, 0.52, 0.40),
    ("bottle",     0.71, 0.40, 0.43, 0.30),
    ("knife",      0.65, 0.00, 0.12, 0.00),
]:
    sf = scorer.compute_sf(ci, sg, ss, sp)
    mark = " ← O*" if label == "wine glass" else ""
    print(f"  {label:<15} Sf = {sf:.4f}{mark}")


## CELL 8 — Visualizer Module

In [ ]:
# ============================================================
# CELL 8 — VISUALIZER MODULE
# ============================================================
# GREEN  box = O* (best selected object)
# YELLOW box = runner-up (#2, #3)
# RED    box = all other detected objects

class Visualizer:
    """ASTRA Result Visualizer — annotates images for DVCon report."""

    C_BEST   = (0, 210, 0)       # Green  (BGR)
    C_RUNNER = (0, 210, 210)     # Yellow (BGR)
    C_OTHER  = (50, 50, 210)     # Red    (BGR)
    C_BG     = (15, 15, 15)      # dark label background

    def draw_results(self, image_rgb, ranked_objects, task, top_n=3):
        img = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

        for i, obj in enumerate(ranked_objects):
            x1, y1, x2, y2 = obj['bbox']
            lbl = obj['label']
            sf  = obj['final_score']
            ci  = obj['confidence']
            sg  = obj['graph_score']

            if i == 0:
                color, thick = self.C_BEST, 3
                tag = f"O*: {lbl}  Sf={sf:.3f}"
            elif i < top_n:
                color, thick = self.C_RUNNER, 2
                tag = f"#{i+1}: {lbl}  Sf={sf:.3f}"
            else:
                color, thick = self.C_OTHER, 1
                tag = f"{lbl}  Ci={ci:.2f}"

            cv2.rectangle(img, (x1, y1), (x2, y2), color, thick)
            (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 1)
            ly = y1 - 8 if y1 > 24 else y2 + th + 8
            cv2.rectangle(img, (x1, ly-th-3), (x1+tw+4, ly+2), self.C_BG, -1)
            cv2.putText(img, tag, (x1+2, ly),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 1, cv2.LINE_AA)

        # Header bar
        hdr = f"Task: {task}  |  O*={ranked_objects[0]['label']}  Sf={ranked_objects[0]['final_score']:.3f}"
        cv2.rectangle(img, (0,0), (img.shape[1], 34), (15,15,15), -1)
        cv2.putText(img, hdr, (8,22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.60, (255,255,255), 1, cv2.LINE_AA)
        return img

    def show(self, bgr, figsize=(14,8)):
        plt.figure(figsize=figsize)
        plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        plt.axis('off'); plt.tight_layout(); plt.show()

    def show_score_table(self, ranked_objects, task):
        sep = "═" * 79
        print(f"\n{sep}")
        print(f"  Task : {task}")
        print(sep)
        print(f"  {'Rank':<5} {'Label':<22} {'Ci':>6} {'Sg':>6} {'Ss':>6} {'Sp':>6} {'Sf':>8}")
        print("─" * 79)
        for i, o in enumerate(ranked_objects):
            mk = " ← O*" if i == 0 else ""
            print(f"  #{i+1:<4} {o['label']:<22}"
                  f" {o['confidence']:>6.3f}"
                  f" {o['graph_score']:>6.3f}"
                  f" {o['semantic_sim']:>6.3f}"
                  f" {o.get('scene_score',0.0):>6.3f}"
                  f" {o['final_score']:>8.4f}{mk}")
        print(f"{sep}\n")

    def save(self, bgr, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        cv2.imwrite(path, bgr)


visualizer = Visualizer()
print("✅ Visualizer ready!")


## CELL 9 — ASTRA Master Pipeline

In [ ]:
# ============================================================
# CELL 9 — ASTRA MASTER PIPELINE
# ============================================================
# Ties all 6 modules together.
# Stages:
#   1. Image preprocessing
#   2. Object detection        (YOLOv5s)
#   3. Task encoding           (CLIP ensemble)
#   4. Object embedding        (CLIP dual-channel)
#   5. Knowledge graph         (Sg)
#   6. Semantic similarity     (Ss = cosine Te·Ei)
#   7. Final scoring           (Sf = α·Ci + β·Sg + γ·Ss)
#   8. Scene context reasoning (boost + re-rank)
#   9. Selection + visualise   (O* = argmax Sf)

class ASTRAPipeline:
    """
    ASTRA end-to-end pipeline for task-aware object selection.
    Input : image (path or numpy RGB) + task text
    Output: O* label, bbox, score, full ranking, latency
    """

    def __init__(self):
        print("Initialising ASTRA pipeline...")
        self.detector    = detector
        self.encoder     = encoder
        self.graph       = kg
        self.scorer      = scorer
        self.scene_ctx   = scene_reasoner
        self.visualizer  = visualizer
        print("✅ ASTRA pipeline ready!\n")

    def run(self, image_input, task: str,
            save_path=None, show=True, verbose=True) -> dict:
        t0 = time.time()

        # Stage 1+2 — detect with fallback
        detected, image_rgb = self.detector.detect_with_fallback(image_input)

        if verbose:
            print(f"[ASTRA] Task : '{task}'")
            print(f"        Found: {[d['label'] for d in detected]}")

        if not detected:
            print("[ASTRA] ⚠ No objects detected even with low threshold.")
            return None

        # Stages 3–8 — score + context
        ranked = self.scorer.rank_objects(
            detected, task, self.graph, self.encoder, self.scene_ctx
        )

        best       = ranked[0]
        latency_ms = (time.time() - t0) * 1000

        if verbose:
            print(f"        O*   : {best['label']}  "
                  f"Sf={best['final_score']:.4f}  "
                  f"(Ci={best['confidence']:.3f}, "
                  f"Sg={best['graph_score']:.3f}, "
                  f"Ss={best['semantic_sim']:.3f}, "
                  f"Sp={best.get('scene_score',0.0):.3f})")
            print(f"        Time : {latency_ms:.0f} ms")

        # Stage 9 — visualise
        annotated = self.visualizer.draw_results(image_rgb, ranked, task)
        if save_path:
            self.visualizer.save(annotated, save_path)
            if verbose: print(f"        Saved: {save_path}")
        if show:
            self.visualizer.show_score_table(ranked, task)
            self.visualizer.show(annotated)

        return {
            'task'           : task,
            'best_label'     : best['label'],
            'best_bbox'      : best['bbox'],
            'best_score'     : best['final_score'],
            'latency_ms'     : round(latency_ms, 1),
            'annotated_image': annotated,
            'all_ranked'     : [
                {k: o[k] for k in
                 ('label','final_score','confidence','graph_score',
                  'semantic_sim','scene_score','context_boost')}
                for o in ranked
            ],
        }


pipeline = ASTRAPipeline()


## CELL 10 — Download & Verify COCO Test Images

In [ ]:
# ============================================================
# CELL 10 — PRETRAINED COCO CLASS-BASED IMAGE STRATEGY
# ============================================================
# Instead of downloading random COCO val2017 images and hoping
# they contain the right objects, this cell:
#
#   1. Defines ALL 80 COCO classes YOLOv5 already knows
#   2. Maps each of the 14 tasks to its primary COCO class
#   3. Uses pycocotools to GUARANTEE the downloaded image
#      actually contains that class in its annotations
#   4. Picks the image with the MOST annotated instances
#      of that class (clearest, most prominent scene)
#
# This means YOLOv5 will ALWAYS find the relevant object
# because the image is proven to contain it.
# ============================================================

import os, json, urllib.request, zipfile
os.makedirs("test_images",       exist_ok=True)
os.makedirs("results/snapshots", exist_ok=True)

# ── ALL 80 COCO CLASSES (exactly as YOLOv5 labels them) ──────
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird',
    'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat',
    'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
    'wine glass', 'cup', 'fork', 'knife', 'spoon',
    'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut',
    'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven',
    'toaster', 'sink', 'refrigerator', 'book', 'clock',
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

print("✅ 80 COCO classes loaded")
print(f"   Total classes : {len(COCO_CLASSES)}")
print()

# ── TASK → PRIMARY COCO CLASS MAPPING ────────────────────────
# Each task maps to the EXACT YOLOv5 class label string.
# Secondary classes are fallbacks used to find richer scenes.

TASK_CLASS_MAP = {
    "step on something to reach top of a shelf": {
        "primary"   : "chair",
        "secondary" : ["bench", "dining table", "couch", "bed"],
        "rationale" : "Chair is the most climbable COCO object"
    },
    "sit comfortably": {
        "primary"   : "couch",
        "secondary" : ["chair", "bench", "bed"],
        "rationale" : "Couch is most comfortable seating in COCO"
    },
    "place flowers": {
        "primary"   : "vase",
        "secondary" : ["cup", "bottle", "bowl", "potted plant"],
        "rationale" : "Vase is the canonical flower container"
    },
    "get potatoes out of fire": {
        "primary"   : "fork",
        "secondary" : ["knife", "spoon", "bowl", "dining table"],
        "rationale" : "Fork is the closest COCO grabbing tool"
    },
    "water a plant": {
        "primary"   : "bottle",
        "secondary" : ["cup", "bowl", "potted plant", "vase"],
        "rationale" : "Bottle is the best COCO watering vessel"
    },
    "get lemon out of tea": {
        "primary"   : "spoon",
        "secondary" : ["fork", "cup", "bowl", "dining table"],
        "rationale" : "Spoon is the ideal small extraction tool"
    },
    "dig a hole": {
        "primary"   : "baseball bat",
        "secondary" : ["knife", "fork", "umbrella", "tennis racket"],
        "rationale" : "Baseball bat is the closest rigid digging tool"
    },
    "open a bottle of beer": {
        "primary"   : "knife",
        "secondary" : ["scissors", "fork", "bottle", "spoon"],
        "rationale" : "Knife is the best bottle-opening tool in COCO"
    },
    "open a parcel": {
        "primary"   : "scissors",
        "secondary" : ["knife", "fork"],
        "rationale" : "Scissors is the canonical parcel-opening tool"
    },
    "serve wine": {
        "primary"   : "wine glass",
        "secondary" : ["cup", "bottle", "bowl", "dining table"],
        "rationale" : "Wine glass is the exact COCO class for this"
    },
    "pour sugar": {
        "primary"   : "spoon",
        "secondary" : ["bowl", "cup", "bottle", "dining table"],
        "rationale" : "Spoon is the sugar-pouring tool in COCO"
    },
    "smear butter": {
        "primary"   : "knife",
        "secondary" : ["spoon", "fork", "dining table", "bowl"],
        "rationale" : "Knife is the spreading tool"
    },
    "extinguish fire": {
        "primary"   : "fire hydrant",
        "secondary" : ["bottle", "bowl"],
        "rationale" : "Fire hydrant is the COCO fire-fighting object"
    },
    "pound a carpet": {
        "primary"   : "baseball bat",
        "secondary" : ["tennis racket", "umbrella", "frisbee"],
        "rationale" : "Baseball bat is the best striking tool in COCO"
    },
}

print("✅ Task-to-COCO-class mapping ready")
print()
print(f"{'Task':<48} {'Primary Class':<18} {'Secondary Count'}")
print("─" * 76)
for task, info in TASK_CLASS_MAP.items():
    print(f"  {task:<46} {info['primary']:<18} {len(info['secondary'])} fallbacks")


# ── STEP 1: Download COCO annotations ────────────────────────
ANN_ZIP  = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANN_DIR  = "coco_annotations"
ANN_FILE = f"{ANN_DIR}/annotations/instances_val2017.json"

if not os.path.exists(ANN_FILE):
    print("\nDownloading COCO val2017 annotations (~240 MB)...")
    zip_path = "ann.zip"
    urllib.request.urlretrieve(ANN_ZIP, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(ANN_DIR)
    os.remove(zip_path)
    print("✅ Annotations downloaded.")
else:
    print("\n✅ COCO annotations already present.")

# ── STEP 2: Load COCO API ─────────────────────────────────────
from pycocotools.coco import COCO
coco = COCO(ANN_FILE)
print(f"✅ COCO API loaded — {len(coco.imgs)} val2017 images available\n")

# ── STEP 3: For each task, find the BEST guaranteed image ─────
# "Best" = image where:
#   a) primary class IS annotated (guaranteed)
#   b) maximum number of secondary class instances present
#      (richest scene = most objects for ASTRA to choose from)

COCO_IMG_URL = "http://images.cocodataset.org/val2017/"

def get_best_image_for_class(primary_class, secondary_classes,
                              top_candidates=5):
    """
    Returns (image_id, filename, n_secondary) for the val2017 image
    that contains the primary class AND the most secondary classes.
    Guaranteed: primary class IS in the image annotations.
    """
    # Get COCO category ID for primary class
    primary_cat_ids = coco.getCatIds(catNms=[primary_class])
    if not primary_cat_ids:
        print(f"    ⚠ '{primary_class}' not found in COCO categories!")
        return None, None, 0

    # Get all val2017 images containing primary class
    img_ids = coco.getImgIds(catIds=primary_cat_ids)
    if not img_ids:
        print(f"    ⚠ No images found for '{primary_class}'")
        return None, None, 0

    # Get secondary category IDs
    sec_cat_ids = []
    for sc in secondary_classes:
        cids = coco.getCatIds(catNms=[sc])
        sec_cat_ids.extend(cids)

    # Score each candidate image by:
    #   - number of secondary class annotations (higher = richer scene)
    #   - number of primary class annotations   (higher = more prominent)
    best_img_id    = None
    best_score     = -1
    best_n_primary = 0
    best_n_sec     = 0

    # Check up to 300 candidates for best scene
    for img_id in img_ids[:300]:
        # Count primary instances
        n_primary = len(coco.getAnnIds(
            imgIds=img_id, catIds=primary_cat_ids
        ))
        # Count secondary instances
        n_sec = len(coco.getAnnIds(
            imgIds=img_id, catIds=sec_cat_ids
        )) if sec_cat_ids else 0

        # Score = weighted combination
        # Primary count matters more (want clear detection)
        score = (n_primary * 2) + n_sec

        if score > best_score:
            best_score     = score
            best_img_id    = img_id
            best_n_primary = n_primary
            best_n_sec     = n_sec

    if best_img_id is None:
        return None, None, 0

    img_info = coco.loadImgs(best_img_id)[0]
    return best_img_id, img_info['file_name'], best_n_sec


def download_image(fname, save_dir="test_images"):
    out_path = os.path.join(save_dir, fname)
    if not os.path.exists(out_path):
        try:
            urllib.request.urlretrieve(COCO_IMG_URL + fname, out_path)
        except Exception as e:
            print(f"    ⚠ Download failed: {e}")
            return None
    return out_path


# ── STEP 4: Find + download best image per task ───────────────
print("Finding best guaranteed images for all 14 tasks...")
print("─" * 76)

downloaded   = []   # (task_id, task_name, local_path)
task_img_map = {}   # task_name → {img_id, fname, n_secondary}

for tid, task in enumerate(TaskObjectGraph.TASK_KEYS, start=1):
    info       = TASK_CLASS_MAP[task]
    primary    = info["primary"]
    secondary  = info["secondary"]
    rationale  = info["rationale"]

    img_id, fname, n_sec = get_best_image_for_class(
        primary, secondary
    )

    if img_id is None:
        print(f"  ❌ T{tid:02d}: No image found for '{primary}'")
        downloaded.append((tid, task, None))
        continue

    local_path = download_image(fname)
    ok         = local_path and os.path.exists(local_path)
    status     = "✅" if ok else "❌"

    # Zero-pad image id for display
    img_id_str = str(img_id).zfill(12)
    print(f"  {status} T{tid:02d}: [{img_id_str}]  "
          f"primary='{primary}'  "
          f"co-objects={n_sec}  "
          f"| {task[:35]}")

    task_img_map[task] = {
        "img_id"     : img_id,
        "fname"      : fname,
        "path"       : local_path,
        "primary"    : primary,
        "n_secondary": n_sec,
        "rationale"  : rationale
    }
    downloaded.append((tid, task, local_path))

n_ok = sum(1 for _, _, p in downloaded if p)
print()
print("═" * 76)
print(f"  Ready: {n_ok}/{len(downloaded)} images — "
      f"all GUARANTEED to contain their primary COCO class")
print("═" * 76)
print()
print("Each image was selected because:")
print("  ✓ Primary object IS annotated in COCO val2017 ground truth")
print("  ✓ YOLOv5 was trained on these exact classes — detection is reliable")
print("  ✓ Image chosen from up to 300 candidates for richest scene")
print()
print("Next: run Cell 11 to test a single task, or Cell 12 for all 14.")

## CELL 11 — Run Single Query (Quick Test)

In [ ]:
# ============================================================
# CELL 11 — RUN SINGLE QUERY (QUICK TEST)
# ============================================================
# Change TASK_ID (1-14) to test any of the 14 official tasks.

TASK_ID = 10   # 1-14  →  10 = "serve wine"

tid, task_name, img_path = downloaded[TASK_ID - 1]

print(f"Task {tid}: {task_name}")
print(f"Image    : {img_path}")
print()

if img_path and os.path.exists(img_path):
    result = pipeline.run(
        image_input = img_path,
        task        = task_name,
        save_path   = f"results/snapshots/task_{tid:02d}_test.jpg",
        show        = True,
        verbose     = True,
    )
    if result:
        print(f"\n🏆  O* = '{result['best_label']}'")
        print(f"    Sf = {result['best_score']:.4f}   |   latency = {result['latency_ms']} ms")
else:
    print("⚠ Image not available. Re-run Cell 10 or set img_path manually.")


## CELL 12 — Run All 14 Tasks (Full Evaluation)

In [ ]:
# ============================================================
# CELL 12 — FULL EVALUATION: ALL 14 COCO-TASKS
# ============================================================
# Ground truth based on COCO-Tasks paper Table 1 + COCO-80 mapping.

GROUND_TRUTH = {
    1 : ["chair", "bench", "couch", "dining table", "bed"],
    2 : ["couch", "chair", "bench", "bed"],
    3 : ["vase", "cup", "wine glass", "bottle"],
    4 : ["fork", "knife", "spoon"],
    5 : ["bottle", "cup", "bowl", "wine glass"],
    6 : ["spoon", "fork"],
    7 : ["baseball bat", "knife", "fork", "umbrella"],
    8 : ["knife", "scissors"],
    9 : ["scissors", "knife"],
    10: ["wine glass", "cup"],
    11: ["spoon", "bowl", "cup"],
    12: ["knife", "spoon"],
    13: ["fire hydrant", "bottle"],
    14: ["baseball bat", "tennis racket", "umbrella"],
}

all_results   = []
correct_count = 0
total_latency = 0.0

print("Running all 14 COCO-Tasks evaluations...")
print("─" * 82)

for task_id, task_name, img_path in downloaded:
    if img_path is None or not os.path.exists(img_path):
        print(f"  ⚠ Task {task_id:2d}: Image missing — skipping.")
        continue

    result = pipeline.run(
        image_input = img_path,
        task        = task_name,
        save_path   = f"results/snapshots/task_{task_id:02d}_{task_name[:18].replace(' ','_')}.jpg",
        show        = False,
        verbose     = False,
    )

    if result is None:
        print(f"  ❌ Task {task_id:2d}: No detections.")
        continue

    expected   = GROUND_TRUTH.get(task_id, [])
    is_correct = result['best_label'] in expected
    correct_count  += int(is_correct)
    total_latency  += result['latency_ms']

    mark = "✅" if is_correct else "❌"
    print(f"  {mark} Task {task_id:2d}: {task_name:<46}"
          f"→ {result['best_label']:<20}"
          f"Sf={result['best_score']:.3f}  {result['latency_ms']:.0f}ms")

    all_results.append({
        'task_id'   : task_id,
        'task'      : task_name,
        'image'     : img_path,
        'best_label': result['best_label'],
        'best_score': result['best_score'],
        'expected'  : expected,
        'correct'   : is_correct,
        'latency_ms': result['latency_ms'],
        'all_ranked': result['all_ranked'],
    })

# ── Save JSON ──
os.makedirs("results", exist_ok=True)
with open("results/evaluation_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

# ── Summary ──
n   = len(all_results)
acc = 100 * correct_count / max(n, 1)
avg = total_latency / max(n, 1)
print()
print("═" * 82)
print(f"  ASTRA Stage 2A — Final Results")
print(f"  Accuracy    : {correct_count}/{n}  ({acc:.1f}%)")
print(f"  Avg Latency : {avg:.0f} ms / image")
print(f"  JSON saved  : results/evaluation_results.json")
print("═" * 82)


In [ ]:
# ============================================================
# CELL 12B — LATENCY + CPU UTILIZATION MEASUREMENT
# ============================================================
# Run this cell IMMEDIATELY after Cell 12 finishes.
# It measures:
#   1. Per-task latency (already logged by pipeline.run())
#   2. Peak CPU % during a live inference run
#   3. RAM usage before and after
#   4. Generates a report-ready summary table
# No extra hardware needed — all CPU metrics via psutil.
# ============================================================

import psutil
import threading
import time
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── PART 1: Extract latency from already-run results ─────────
# all_results was populated by Cell 12 — use it directly.

if not all_results:
    print("⚠ Run Cell 12 first to populate all_results.")
else:
    latencies  = [r['latency_ms'] for r in all_results]
    task_names = [r['task'][:30] + '…' if len(r['task']) > 30
                  else r['task'] for r in all_results]
    task_ids   = [r['task_id'] for r in all_results]

    avg_lat  = sum(latencies) / len(latencies)
    min_lat  = min(latencies)
    max_lat  = max(latencies)
    med_lat  = sorted(latencies)[len(latencies)//2]

    print("═" * 60)
    print("  LATENCY REPORT — All 14 Tasks")
    print("═" * 60)
    print(f"  {'Task':<35} {'Latency (ms)':>12}")
    print("─" * 60)
    for i, r in enumerate(all_results):
        bar = '█' * int(latencies[i] / 20)   # 1 block = 20ms
        print(f"  T{r['task_id']:02d} {r['task'][:30]:<30} "
              f"{latencies[i]:>8.0f} ms  {bar}")
    print("─" * 60)
    print(f"  {'Average':<35} {avg_lat:>8.0f} ms")
    print(f"  {'Minimum':<35} {min_lat:>8.0f} ms")
    print(f"  {'Maximum':<35} {max_lat:>8.0f} ms")
    print(f"  {'Median':<35} {med_lat:>8.0f} ms")
    print(f"  {'Target (proposal)':<35} {'< 50 ms':>12}")
    print(f"  {'Status':<35} "
          f"{'✅ Met' if avg_lat < 50 else '⚠ Exceeds (CPU-only, Stage 2A)':>12}")
    print("═" * 60)


# ── PART 2: Live CPU monitoring during one inference ─────────
# Spawns a background thread that samples CPU % every 100ms
# while pipeline.run() executes on the main thread.
# Gives accurate peak and average CPU usage during inference.

class CPUMonitor:
    """Background thread that samples CPU utilization."""
    def __init__(self, interval_ms=100):
        self.interval   = interval_ms / 1000
        self.samples    = []
        self.ram_before = psutil.virtual_memory().used / 1e6   # MB
        self._stop      = threading.Event()

    def start(self):
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def _run(self):
        while not self._stop.is_set():
            self.samples.append(psutil.cpu_percent(interval=None))
            time.sleep(self.interval)

    def stop(self):
        self._stop.set()
        self._thread.join()
        self.ram_after = psutil.virtual_memory().used / 1e6    # MB
        return self

    @property
    def peak_cpu(self):
        return max(self.samples) if self.samples else 0

    @property
    def avg_cpu(self):
        return sum(self.samples)/len(self.samples) if self.samples else 0

    @property
    def ram_delta_mb(self):
        return self.ram_after - self.ram_before


print("\nRunning live CPU monitoring during one inference...")
print("(Using Task 10: serve wine — a typical mid-complexity task)")

# Pick Task 10 image (serve wine) from downloaded list
task10 = next(
    ((tid, task, path) for tid, task, path in downloaded
     if tid == 10 and path and os.path.exists(path)),
    None
)

if task10 is None:
    print("⚠ Task 10 image not found. Re-run Cell 10.")
else:
    tid, task_name, img_path = task10

    # Warm up psutil (first call is always 0.0)
    psutil.cpu_percent(interval=None)
    time.sleep(0.1)

    monitor = CPUMonitor(interval_ms=100).start()

    # ── Run the full pipeline (this is what we're measuring) ──
    result = pipeline.run(
        image_input = img_path,
        task        = task_name,
        save_path   = None,
        show        = False,
        verbose     = False,
    )

    monitor.stop()

    cpu_samples = monitor.samples
    peak_cpu    = monitor.peak_cpu
    avg_cpu     = monitor.avg_cpu
    ram_delta   = monitor.ram_delta_mb
    ram_total   = psutil.virtual_memory().total / 1e9   # GB
    ram_used    = psutil.virtual_memory().used  / 1e9   # GB
    cpu_count   = psutil.cpu_count(logical=True)
    cpu_phys    = psutil.cpu_count(logical=False)
    cpu_freq    = psutil.cpu_freq()

    print()
    print("═" * 60)
    print("  CPU UTILIZATION REPORT")
    print("═" * 60)
    print(f"  Logical CPUs        : {cpu_count}")
    print(f"  Physical cores      : {cpu_phys}")
    if cpu_freq:
        print(f"  CPU frequency       : {cpu_freq.current:.0f} MHz")
    print(f"  Peak CPU during inf : {peak_cpu:.1f} %")
    print(f"  Avg CPU during inf  : {avg_cpu:.1f} %")
    print(f"  RAM — total         : {ram_total:.1f} GB")
    print(f"  RAM — used (current): {ram_used:.1f} GB")
    print(f"  RAM — delta (infer.): {ram_delta:+.1f} MB")
    print(f"  Samples collected   : {len(cpu_samples)}")
    print(f"  Inference latency   : {result['latency_ms']:.0f} ms")
    print("─" * 60)
    print("  FPGA utilization    : N/A (Stage 3 — hardware not yet")
    print("                        deployed; will be measured after")
    print("                        Genesys-2 integration in Stage 3)")
    print("═" * 60)


# ── PART 3: Generate report-ready charts ─────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("ASTRA Stage 2A — Performance Metrics",
             fontsize=13, fontweight='bold')

# Chart 1 — Per-task latency bar chart
ax1 = axes[0]
colors = ['#97C459' if l < 50 else '#EF9F27' if l < 150 else '#E24B4A'
          for l in latencies]
bars = ax1.barh(task_ids, latencies, color=colors, height=0.6)
ax1.axvline(50,  color='#3B6D11', linestyle='--', linewidth=1,
            label='Target 50ms')
ax1.axvline(avg_lat, color='#185FA5', linestyle='-', linewidth=1.5,
            label=f'Avg {avg_lat:.0f}ms')
ax1.set_xlabel("Latency (ms)", fontsize=10)
ax1.set_ylabel("Task ID", fontsize=10)
ax1.set_title("Inference Latency per Task", fontsize=11, fontweight='bold')
ax1.set_yticks(task_ids)
ax1.set_yticklabels([f"T{t}" for t in task_ids], fontsize=9)
ax1.legend(fontsize=8)
ax1.grid(axis='x', alpha=0.3)

# Chart 2 — CPU usage over time during inference
if task10:
    ax2 = axes[1]
    t_axis = [i * 0.1 for i in range(len(cpu_samples))]
    ax2.fill_between(t_axis, cpu_samples, alpha=0.3, color='#378ADD')
    ax2.plot(t_axis, cpu_samples, color='#185FA5', linewidth=1.5)
    ax2.axhline(avg_cpu, color='#854F0B', linestyle='--', linewidth=1,
                label=f'Avg {avg_cpu:.1f}%')
    ax2.axhline(peak_cpu, color='#E24B4A', linestyle='--', linewidth=1,
                label=f'Peak {peak_cpu:.1f}%')
    ax2.set_xlabel("Time (seconds)", fontsize=10)
    ax2.set_ylabel("CPU utilization (%)", fontsize=10)
    ax2.set_title("CPU Usage During Inference\n(Task 10: serve wine)",
                  fontsize=11, fontweight='bold')
    ax2.set_ylim(0, 105)
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

# Chart 3 — Summary metrics comparison table as visual
ax3 = axes[2]
ax3.axis('off')
metrics = [
    ["Metric",               "Value",           "Target"],
    ["Avg latency",          f"{avg_lat:.0f} ms", "< 50 ms"],
    ["Min latency",          f"{min_lat:.0f} ms", "—"],
    ["Max latency",          f"{max_lat:.0f} ms", "—"],
    ["Peak CPU",             f"{peak_cpu:.1f} %", "< 100 %"],
    ["Avg CPU",              f"{avg_cpu:.1f} %",  "—"],
    ["RAM delta",            f"{ram_delta:+.0f} MB", "—"],
    ["Inference device",     "CPU",               "CPU (Stage 2A)"],
    ["FPGA utilization",     "N/A",               "Stage 3"],
]
tbl = ax3.table(
    cellText  = metrics[1:],
    colLabels = metrics[0],
    loc       = 'center',
    cellLoc   = 'left'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.5)
# Highlight header
for j in range(3):
    tbl[0, j].set_facecolor('#E6F1FB')
    tbl[0, j].set_text_props(fontweight='bold')
ax3.set_title("Performance Summary", fontsize=11, fontweight='bold', pad=12)

plt.tight_layout()
perf_chart_path = "results/ASTRA_performance_metrics.jpg"
os.makedirs("results", exist_ok=True)
plt.savefig(perf_chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Performance chart saved → {perf_chart_path}")

# ── PART 4: Save metrics to JSON for report ──────────────────
metrics_dict = {
    "latency": {
        "avg_ms"       : round(avg_lat, 1),
        "min_ms"       : round(min_lat, 1),
        "max_ms"       : round(max_lat, 1),
        "median_ms"    : round(med_lat, 1),
        "target_ms"    : 50,
        "per_task"     : [
            {"task_id": r['task_id'],
             "task"   : r['task'],
             "ms"     : round(r['latency_ms'], 1)}
            for r in all_results
        ]
    },
    "cpu": {
        "peak_pct"     : round(peak_cpu, 1),
        "avg_pct"      : round(avg_cpu,  1),
        "logical_cores": cpu_count,
        "physical_cores": cpu_phys,
        "freq_mhz"     : round(cpu_freq.current, 0) if cpu_freq else "N/A",
        "ram_total_gb" : round(ram_total, 1),
        "ram_used_gb"  : round(ram_used,  1),
        "ram_delta_mb" : round(ram_delta, 1),
    },
    "fpga": {
        "status"       : "not implemented — Stage 3",
        "note"         : "Will be measured after Genesys-2 deployment"
    },
    "inference_device": "CPU",
    "stage"           : "2A"
}

with open("results/performance_metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

print(f"✅ Metrics JSON saved → results/performance_metrics.json")
print()
print("Use these numbers directly in your Stage 2A report.")

## CELL 13 — Generate Report Snapshot Grid

In [ ]:
# ============================================================
# CELL 13 — GENERATE REPORT SNAPSHOT GRID (Page 2 of report)
# ============================================================
snapshot_dir = "results/snapshots"
snaps = sorted([
    os.path.join(snapshot_dir, f)
    for f in os.listdir(snapshot_dir) if f.endswith('.jpg')
])

if not snaps:
    print("No snapshots found. Run Cell 12 first.")
else:
    ncols = 3
    nrows = math.ceil(len(snaps) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 5))
    fig.suptitle("ASTRA — Stage 2A: All 14 Task Results",
                 fontsize=16, fontweight='bold', y=1.01)
    axes = axes.flatten()

    for i, p in enumerate(snaps):
        bgr = cv2.imread(p)
        if bgr is None: continue
        axes[i].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        axes[i].set_title(os.path.basename(p).replace('.jpg','').replace('_',' '), fontsize=8)
        axes[i].axis('off')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    grid_path = "results/ASTRA_results_grid.jpg"
    plt.savefig(grid_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✅ Grid saved → {grid_path}")


## CELL 14 — Accuracy Summary Table (Report-Ready)

In [ ]:
# ============================================================
# CELL 14 — ACCURACY SUMMARY TABLE
# ============================================================
if not all_results:
    print("No results. Run Cell 12 first.")
else:
    n_eval = len(all_results)
    n_corr = sum(r['correct'] for r in all_results)
    acc    = 100 * n_corr / n_eval
    avg_l  = sum(r['latency_ms'] for r in all_results) / n_eval

    print("\n" + "═"*88)
    print(f"  {'ID':>3}  {'Task':<46} {'Selected O*':<22} {'OK':>4}   {'Sf':>6}")
    print("─"*88)
    for r in sorted(all_results, key=lambda x: x['task_id']):
        mk = "✅" if r['correct'] else "❌"
        print(f"  {r['task_id']:>3}  {r['task']:<46}"
              f" {r['best_label']:<22} {mk:>4}  {r['best_score']:>6.3f}")
    print("═"*88)
    print(f"  Accuracy: {n_corr}/{n_eval} = {acc:.1f}%     Avg latency: {avg_l:.0f} ms")
    print("═"*88)

    # ── Matplotlib export table ──
    fig, ax = plt.subplots(figsize=(15, 0.45 * n_eval + 1.8))
    ax.axis('off')
    rows = [
        [r['task_id'],
         r['task'][:43]+('…' if len(r['task'])>43 else ''),
         r['best_label'],
         "✓" if r['correct'] else "✗",
         f"{r['best_score']:.3f}"]
        for r in sorted(all_results, key=lambda x: x['task_id'])
    ]
    rows.append(["", f"Accuracy: {n_corr}/{n_eval} = {acc:.1f}%", "", "", f"μ={avg_l:.0f}ms"])
    tbl = ax.table(cellText=rows,
                   colLabels=["ID","Task","Selected O*","✓/✗","Sf"],
                   loc='center', cellLoc='left')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.45)
    plt.title("ASTRA Stage 2A — Evaluation Results", fontsize=13, fontweight='bold', pad=14)
    plt.tight_layout()
    tbl_path = "results/ASTRA_results_table.jpg"
    plt.savefig(tbl_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Table saved → {tbl_path}")


## CELL 15 — Upload Your Own Image (Custom Test)

In [ ]:
# ============================================================
# CELL 15 — UPLOAD YOUR OWN IMAGE
# ============================================================
from google.colab import files

print("Upload any image to test ASTRA on a custom scene.")
uploaded_files = files.upload()

if uploaded_files:
    img_path = list(uploaded_files.keys())[0]
    print(f"Uploaded: {img_path}")

    # ← Change to any of the 14 official task strings
    MY_TASK = "serve wine"

    result = pipeline.run(
        image_input = img_path,
        task        = MY_TASK,
        save_path   = f"results/snapshots/custom_{MY_TASK.replace(' ','_')}.jpg",
        show        = True,
        verbose     = True,
    )
    if result:
        print(f"\n🏆 O* = '{result['best_label']}'  Sf={result['best_score']:.4f}")
else:
    print("No file uploaded.")


## CELL 16 — Download All Results

In [ ]:
# ============================================================
# CELL 16 — DOWNLOAD ALL RESULTS (for report submission)
# ============================================================
# Zips: snapshots/, evaluation_results.json, grid, table
from google.colab import files

zip_name = "ASTRA_Stage2A_Results.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("results"):
        for fname in fnames:
            fp = os.path.join(root, fname)
            zf.write(fp, fp)

print(f"✅ Zipped → {zip_name}")
files.download(zip_name)
print("Download started!")


---
## Architecture Summary

```
INPUT: Image (I)  +  Task Description (T)
           │
    ┌──────▼──────────────────────────────────────────────┐
    │  Stage 1: Image Preprocessing                        │
    │  BGR→RGB · resize · normalise                        │
    └──────────────────────────┬──────────────────────────┘
                               │
    ┌──────────────────────────▼──────────────────────────┐
    │  Stage 2: Object Detection  (YOLOv5s, CPU)           │
    │  Output: {label, Ci, bbox, crop}  per object         │
    │  Fallback: retry with conf=0.10 if 0 detections      │
    └──────────────────────────┬──────────────────────────┘
            │                  │
    ┌───────▼──────┐           │
    │  Stage 3:    │           │
    │  Task encode │           │
    │  T → Te 512d │           │
    └───────┬──────┘           │
            │                  │
    ┌───────▼──────────────────▼──────────────────────────┐
    │  Stage 4: Object Embedding  (CLIP dual-channel)      │
    │  Ei = avg(encode_label_text, encode_image_crop)      │
    └──────────────────────────┬──────────────────────────┘
                               │
    ┌──────────────────────────▼──────────────────────────┐
    │  Stage 5: Knowledge Graph                            │
    │  task → COCO object → Sg (weight 0.2–1.0)           │
    └──────────────────────────┬──────────────────────────┘
                               │
    ┌──────────────────────────▼──────────────────────────┐
    │  Stage 6: Semantic Similarity                        │
    │  Ss = cosine(Te, Ei)  ∈ [-1, 1]                     │
    └──────────────────────────┬──────────────────────────┘
                               │
    ┌──────────────────────────▼──────────────────────────┐
    │  Stage 7: Final Scoring                              │
    │  Sf = 0.20·Ci + 0.40·Sg + 0.25·Ss + 0.15·Sp        │
    └──────────────────────────┬──────────────────────────┘
                               │
    ┌──────────────────────────▼──────────────────────────┐
    │  Stage 8: Scene Context (Sp) → already in Sf formula │
    │  Fallback boost + co-presence bonus → Sp ∈ [0,1]    │
    └──────────────────────────┬──────────────────────────┘
                               │
    ┌──────────────────────────▼──────────────────────────┐
    │  Stage 9: Selection + Visualise                      │
    │  O* = argmax(Sf)  →  bbox + score + annotated image  │
    └──────────────────────────────────────────────────────┘

OUTPUT: O* label · bounding box · Sf · latency · ranked list
```

**Weight rationale (α=0.20, β=0.40, γ=0.25, δ=0.15):**
- `β` highest → knowledge graph is most task-specific signal
- `γ` = 0.25 → CLIP semantic similarity adds neural reasoning
- `δ` = 0.15 → scene proximity (Sp) captures co-presence + fallback context
- `α` = 0.20 → detection confidence validates real object presence

**Changes from v1 notebook:**
- Cell 6: **Scene Context Reasoning Module added** (was missing in v1)
- Cell 4: **detect_with_fallback()** prevents silent failures on difficult images
- Cell 10: **pycocotools-verified image IDs** (v1 had unverified hardcoded IDs)
- Cell 5: **CLIPEncoder refactored** — get_task_embedding() and get_object_embedding() encapsulate dual-channel logic cleanly
- Confidence threshold **lowered to 0.20** from 0.25 for better recall
